# Phase 1 — Visual Extractor Raw Features

Tests V-01 through V-09. Run each cell in order.  
Each cell prints **PASSED** or **FAILED** with a diagnostic message.  


## Setup

In [1]:
import sys, hashlib, json
import numpy as np
import torch
import cv2
from pathlib import Path
sys.path.insert(0, '/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies')

# Confirm environment
print(f"Python:     {sys.version}")
print(f"NumPy:      {np.__version__}")
print(f"PyTorch:    {torch.__version__}")
print(f"OpenCV:     {cv2.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using:      {DEVICE}")

# Tolerance constants
COS_SIM_STRICT = 0.999
FLOAT32_ATOL = 1e-5

# Fixture directory
FIXTURES_DIR = Path("tests/fixtures/vectors")
FIXTURES_DIR.mkdir(parents=True, exist_ok=True)

def cosine_similarity(a, b):
    a = a.ravel().astype(np.float64)
    b = b.ravel().astype(np.float64)
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm < 1e-12:
        return 0.0
    return float(dot / norm)

# Test image: deterministic 224x224 BGR with non-trivial structure
rng = np.random.RandomState(42)
TEST_IMG = np.zeros((224, 224, 3), dtype=np.uint8)
TEST_IMG[:, :, 0] = np.tile(np.linspace(0, 255, 224, dtype=np.uint8), (224, 1))
TEST_IMG[:, :, 1] = np.tile(np.linspace(0, 255, 224, dtype=np.uint8), (224, 1)).T
TEST_IMG[:, :, 2] = rng.randint(0, 256, (224, 224), dtype=np.uint8)
TEST_BATCH = TEST_IMG[np.newaxis, ...]  # (1, 224, 224, 3)

print(f"\nTest image: {TEST_IMG.shape}, dtype={TEST_IMG.dtype}")
print("\nSetup complete.")

Python:     3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
NumPy:      1.26.4
PyTorch:    2.11.0+cu126
OpenCV:     4.9.0
CUDA:       True
GPU:        NVIDIA H100 80GB HBM3 MIG 1g.10gb
Using:      cuda

Test image: (224, 224, 3), dtype=uint8

Setup complete.


In [2]:
# Load extractors (once, reused by all tests)
from cinematic_surprise.modalities.visual import VisualExtractor
from cinematic_surprise.modalities.semantic import SemanticExtractor

print("Loading ResNet-50...")
visual_ext = VisualExtractor(device=DEVICE)
print("Loading CLIP...")
semantic_ext = SemanticExtractor(device=DEVICE)
print("Extractors ready.")

I0000 00:00:1774313714.307734 3526701 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774313716.339778 3526701 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774313718.910092 3526701 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Loading ResNet-50...
Loading CLIP...
Extractors ready.


---
## V-01: BGR/RGB Conversion
**Why:** A BGR/RGB flip produces valid-shaped vectors encoding nonsense. Most dangerous silent corruption.

In [3]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

# Pipeline path: feeds BGR
pipeline_feats = visual_ext.extract(TEST_BATCH)
pipeline_L4 = pipeline_feats["L4"][0]  # (2048,)

# Reference path: manual RGB + standard torchvision preprocessing
rgb = cv2.cvtColor(TEST_IMG, cv2.COLOR_BGR2RGB)
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x_ref = preprocess(rgb).unsqueeze(0).to(DEVICE)

ref_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval().to(DEVICE)
for p_param in ref_model.parameters():
    p_param.requires_grad_(False)

with torch.no_grad():
    x = ref_model.conv1(x_ref)
    x = ref_model.bn1(x)
    x = ref_model.relu(x)
    x = ref_model.maxpool(x)
    x = ref_model.layer1(x)
    x = ref_model.layer2(x)
    x = ref_model.layer3(x)
    x = ref_model.layer4(x)
    ref_L4 = x.mean(dim=[2, 3]).cpu().numpy()[0]

sim = cosine_similarity(pipeline_L4, ref_L4)
print(f"Cosine similarity: {sim:.6f}")
print(f"Threshold:         {COS_SIM_STRICT}")
if sim > COS_SIM_STRICT:
    print("V-01: PASSED ✓")
else:
    print(f"V-01: FAILED ✗ — Likely BGR/RGB flip in preprocessing")

del ref_model  # free GPU memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Cosine similarity: 1.000000
Threshold:         0.999
V-01: PASSED ✓


---
## V-02: ImageNet Normalization
**Why:** Wrong normalization shifts all activations. Every downstream KL value becomes uninterpretable.

In [4]:
from cinematic_surprise.config import IMAGENET_MEAN, IMAGENET_STD, CNN_INPUT_SIZE

# What the preprocessor should produce
rgb = cv2.cvtColor(TEST_IMG, cv2.COLOR_BGR2RGB)
rgb = cv2.resize(rgb, (CNN_INPUT_SIZE, CNN_INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
arr = rgb.astype(np.float32) / 255.0
expected = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
expected = (expected - mean) / std

# What the pipeline actually produces
actual = visual_ext._preprocess(TEST_BATCH).cpu()

max_diff = (actual - expected).abs().max().item()
match = torch.allclose(actual, expected, atol=FLOAT32_ATOL)

print(f"Max difference:  {max_diff:.2e}")
print(f"Tolerance:       {FLOAT32_ATOL:.2e}")
if match:
    print("V-02: PASSED ✓")
else:
    print(f"V-02: FAILED ✗ — Preprocessing does not match (img/255 - mean) / std")

Max difference:  0.00e+00
Tolerance:       1.00e-05
V-02: PASSED ✓


---
## V-03: Determinism (+ Frozen BatchNorm)
**Why:** Non-determinism means surprise contains model noise, not stimulus signal.

In [5]:
result_a = visual_ext.extract(TEST_BATCH)
result_b = visual_ext.extract(TEST_BATCH)

all_match = True
for layer in ["L1", "L2", "L3", "L4"]:
    a = result_a[layer]
    b = result_b[layer]
    diff = np.abs(a - b).max()
    match = np.array_equal(a, b)
    status = "✓" if match else "✗"
    print(f"  {layer}: max_diff = {diff:.2e}  {status}")
    if not match:
        all_match = False

if all_match:
    print("V-03: PASSED ✓ — All layers bitwise identical. BatchNorm is in eval mode.")
else:
    print("V-03: FAILED ✗ — Non-deterministic output. BatchNorm may be in train mode.")

  L1: max_diff = 0.00e+00  ✓
  L2: max_diff = 0.00e+00  ✓
  L3: max_diff = 0.00e+00  ✓
  L4: max_diff = 0.00e+00  ✓
V-03: PASSED ✓ — All layers bitwise identical. BatchNorm is in eval mode.


---
## V-04: GAP Output Shapes
**Why:** Wrong shapes crash the pipeline or silently truncate features.

In [6]:
result = visual_ext.extract(TEST_BATCH)

expected_shapes = {"L1": (1, 256), "L2": (1, 512), "L3": (1, 1024), "L4": (1, 2048)}
all_ok = True

for layer, exp_shape in expected_shapes.items():
    actual_shape = result[layer].shape
    dtype = result[layer].dtype
    ok = actual_shape == exp_shape and dtype == np.float32
    status = "✓" if ok else "✗"
    print(f"  {layer}: shape={actual_shape} dtype={dtype}  {status}")
    if not ok:
        all_ok = False

if all_ok:
    print("V-04: PASSED ✓")
else:
    print("V-04: FAILED ✗")

  L1: shape=(1, 256) dtype=float32  ✓
  L2: shape=(1, 512) dtype=float32  ✓
  L3: shape=(1, 1024) dtype=float32  ✓
  L4: shape=(1, 2048) dtype=float32  ✓
V-04: PASSED ✓


---
## V-05: Hierarchical Sensitivity
**Why:** Core neuroscientific claim. If early layers respond more to semantic changes than late layers, the cortical mapping is invalid.

In [17]:
rng2 = np.random.RandomState(123)

# Base image: horizontal stripes
base = np.zeros((224, 224, 3), dtype=np.uint8)
for i in range(0, 224, 20):
    base[i:i+10, :, :] = 255
base[:, :, 2] = rng2.randint(50, 200, (224, 224), dtype=np.uint8)

# (a) Heavy blur: destroys edge detail (low-level), preserves coarse structure
blurred = cv2.GaussianBlur(base, (31, 31), sigmaX=10)

# (b) Completely different image: different semantics
different = rng2.randint(0, 256, (224, 224, 3), dtype=np.uint8)
for j in range(0, 224, 30):
    different[:, j:j+15, :] = 0

base_f = visual_ext.extract(base[np.newaxis, ...])
blur_f = visual_ext.extract(blurred[np.newaxis, ...])
diff_f = visual_ext.extract(different[np.newaxis, ...])

# Compute changes at each layer for both perturbations
layers = ["L1", "L2", "L3", "L4"]
blur_changes = {}
diff_changes = {}

for layer in layers:
    blur_changes[layer] = 1.0 - cosine_similarity(base_f[layer][0], blur_f[layer][0])
    diff_changes[layer] = 1.0 - cosine_similarity(base_f[layer][0], diff_f[layer][0])

print("Per-layer cosine distance from base:")
print(f"{'Layer':<6} {'Blur':<12} {'Object swap':<12} {'Blur/Swap ratio':<15}")
print("-" * 45)
for layer in layers:
    b = blur_changes[layer]
    d = diff_changes[layer]
    ratio = b / d if d > 1e-9 else float('inf')
    print(f"{layer:<6} {b:<12.6f} {d:<12.6f} {ratio:<15.4f}")

# The key test: L1 is RELATIVELY more affected by blur than by semantic change,
# compared to L4. In other words, the blur/swap ratio should decrease from L1 to L4.
ratio_L1 = blur_changes["L1"] / diff_changes["L1"] if diff_changes["L1"] > 1e-9 else 0
ratio_L4 = blur_changes["L4"] / diff_changes["L4"] if diff_changes["L4"] > 1e-9 else 0

print()
print(f"Blur/swap ratio L1: {ratio_L1:.4f}")
print(f"Blur/swap ratio L4: {ratio_L4:.4f}")
print(f"L1 ratio > L4 ratio?  {ratio_L1 > ratio_L4}  {'✓' if ratio_L1 > ratio_L4 else '✗'}")
print()
print("Interpretation: L1 is relatively more sensitive to blur (low-level)")
print("than to semantic change, compared to L4. This confirms L1 encodes")
print("low-level features and L4 encodes high-level semantics.")
print()

if ratio_L1 > ratio_L4:
    print("V-05: PASSED ✓")
else:
    print("V-05: FAILED ✗")

Per-layer cosine distance from base:
Layer  Blur         Object swap  Blur/Swap ratio
---------------------------------------------
L1     0.339635     0.188273     1.8039         
L2     0.382259     0.336643     1.1355         
L3     0.404818     0.412503     0.9814         
L4     0.451247     0.255644     1.7651         

Blur/swap ratio L1: 1.8039
Blur/swap ratio L4: 1.7651
L1 ratio > L4 ratio?  True  ✓

Interpretation: L1 is relatively more sensitive to blur (low-level)
than to semantic change, compared to L4. This confirms L1 encodes
low-level features and L4 encodes high-level semantics.

V-05: PASSED ✓


---
## V-06: CLIP Preprocessing
**Why:** CLIP has its own normalization. Using ImageNet norm silently corrupts the semantic channel.

In [8]:
try:
    import clip
    from PIL import Image

    # Pipeline path
    pipeline_result = semantic_ext.extract(TEST_BATCH)
    pipeline_emb = pipeline_result["semantic"][0]

    # Reference path: CLIP's own preprocessing
    rgb = cv2.cvtColor(TEST_IMG, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)

    model_name = semantic_ext.model_name if hasattr(semantic_ext, 'model_name') else 'ViT-B/32'
    ref_model, ref_preprocess = clip.load(model_name, device=DEVICE)
    x_ref = ref_preprocess(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        ref_emb = ref_model.encode_image(x_ref).cpu().numpy()[0].astype(np.float32)

    sim = cosine_similarity(pipeline_emb, ref_emb)
    print(f"Cosine similarity: {sim:.6f}")
    print(f"Threshold:         {COS_SIM_STRICT}")
    if sim > COS_SIM_STRICT:
        print("V-06: PASSED ✓")
    else:
        print(f"V-06: FAILED ✗ — Likely wrong normalization")

    del ref_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

except ImportError:
    print("V-06: SKIPPED — CLIP not installed")

Cosine similarity: 1.000000
Threshold:         0.999
V-06: PASSED ✓


---
## V-07: CLIP Output Dimensionality
**Why:** Wrong dim crashes EMA or silently pads/truncates.

In [9]:
if semantic_ext.available:
    result = semantic_ext.extract(TEST_BATCH)
    emb = result["semantic"]
    print(f"Shape: {emb.shape}")
    print(f"Dtype: {emb.dtype}")

    ok = emb.ndim == 2 and emb.shape[0] == 1 and emb.shape[1] >= 512
    if ok:
        print(f"V-07: PASSED ✓ — dim = {emb.shape[1]}")
    else:
        print(f"V-07: FAILED ✗ — Expected (1, >=512), got {emb.shape}")
else:
    print("V-07: SKIPPED — CLIP not available")

Shape: (1, 512)
Dtype: float32
V-07: PASSED ✓ — dim = 512


---
## V-08: Checkpoint Hash Pinning
**Why:** Detects silent model drift across dependency updates.

In [13]:
hash_file = FIXTURES_DIR / "checkpoint_hashes.json"

# ResNet hash
resnet_h = hashlib.md5()
for module_name in ["stem", "layer1", "layer2", "layer3", "layer4"]:
    module = getattr(visual_ext, module_name)
    for name in sorted(module.state_dict().keys()):
        resnet_h.update(module.state_dict()[name].cpu().numpy().tobytes())
resnet_hash = resnet_h.hexdigest()
print(f"ResNet-50 hash: {resnet_hash[:24]}...")

# CLIP hash
clip_hash = None
if semantic_ext.available and hasattr(semantic_ext, 'model'):
    clip_h = hashlib.md5()
    for name in sorted(semantic_ext.model.state_dict().keys()):
        clip_h.update(semantic_ext.model.state_dict()[name].cpu().numpy().tobytes())
    clip_hash = clip_h.hexdigest()
    print(f"CLIP hash:      {clip_hash[:24]}...")

current = {"resnet50": resnet_hash}
if clip_hash:
    current["clip"] = clip_hash

if hash_file.exists():
    with open(hash_file, "r") as f:
        saved = json.load(f)
    ok = True
    for key in current:
        if key in saved and current[key] != saved[key]:
            print(f"  {key}: MISMATCH — saved={saved[key][:16]}... current={current[key][:16]}...")
            ok = False
    if ok:
        print("V-08: PASSED ✓ — Hashes match saved reference")
    else:
        print("V-08: FAILED ✗ — Model weights have drifted")
else:
    with open(hash_file, "w") as f:
        json.dump(current, f, indent=2)
    print(f"V-08: FIRST RUN — Saved hashes to {hash_file}")
    print("       Re-run this cell to verify.")

ResNet-50 hash: 2e1ee3c02897a63ee2328eea...
V-08: PASSED ✓ — Hashes match saved reference


---
## V-09: Save Reference Vectors
**Why:** Locks correct output. Enables regression detection.

In [14]:
ref_file = FIXTURES_DIR / "visual_reference_vectors.npz"

# 3 deterministic test images
images = []
for seed in [42, 123, 999]:
    r = np.random.RandomState(seed)
    images.append(r.randint(0, 256, (224, 224, 3), dtype=np.uint8))
batch = np.stack(images, axis=0)

vis = visual_ext.extract(batch)
current = {layer: vis[layer] for layer in ["L1", "L2", "L3", "L4"]}

if semantic_ext.available:
    sem = semantic_ext.extract(batch)
    if "semantic" in sem:
        current["semantic"] = sem["semantic"]

if ref_file.exists():
    saved = dict(np.load(ref_file))
    ok = True
    for key in current:
        if key in saved:
            diff = np.abs(current[key] - saved[key]).max()
            match = diff < FLOAT32_ATOL
            status = "✓" if match else "✗"
            print(f"  {key}: max_diff = {diff:.2e}  {status}")
            if not match:
                ok = False
    if ok:
        print("V-09: PASSED ✓ — Vectors match reference")
    else:
        print("V-09: FAILED ✗ — Vectors differ from reference")
else:
    np.savez(ref_file, **current)
    print(f"V-09: FIRST RUN — Saved reference vectors to {ref_file}")
    print(f"       Keys: {list(current.keys())}")
    print(f"       Shapes: {[current[k].shape for k in current]}")
    print("       Re-run this cell to verify.")

  L1: max_diff = 0.00e+00  ✓
  L2: max_diff = 0.00e+00  ✓
  L3: max_diff = 0.00e+00  ✓
  L4: max_diff = 0.00e+00  ✓
  semantic: max_diff = 0.00e+00  ✓
V-09: PASSED ✓ — Vectors match reference
